# 99. Custom Filtration Family (Trigrade, $P=3$)

This notebook shows the full workflow for adding a **custom multigrade filtration family** and using it with the standard pipeline:

1. define a typed filtration object,
2. register family metadata and schema,
3. run `encode(...)` from both typed and `FiltrationSpec` paths,
4. compute an invariant and transform it into an ML-ready feature vector.


## Why this contract exists

For custom families, we separate responsibilities:

- `filtration_kind(::Type{MyFiltration})`: stable symbolic identity.
- `filtration_arity(filtration, data)`: persistence parameter count (`3` here).
- `build_graded_complex(data, filtration; cache=...)`: construction map
  \
  data -> (GradedComplex, axes, orientation).

Registration (`register_filtration_family!`) adds:

- `FiltrationSpec(kind=:my_kind, ...) -> typed filtration` roundtrip,
- schema validation (`required`, `defaults`, `types`, `checks`),
- discovery helpers (`available_filtrations`, `filtration_signature`, `filtration_parameters`).


In [ ]:
using PosetModules
using PosetModules.DataIngestion
using SparseArrays
using Random

Random.seed!(11)


## Step 1 — Define a typed filtration family

We encode three scalar coordinates per simplex (here only vertices), so this family is genuinely trigraded.

Parameters:

- `center`: required, used in the third coordinate,
- `shift`: optional, defaulted by schema,
- `scale`: optional positive multiplier, defaulted by schema.


In [ ]:
struct LectureTrigradeFiltration{P<:NamedTuple} <: AbstractFiltration
    params::P
end

LectureTrigradeFiltration(; center::Real,
                          shift::Real=0.0,
                          scale::Real=1.0,
                          construction::ConstructionOptions=ConstructionOptions()) =
    LectureTrigradeFiltration((;
        center=Float64(center),
        shift=Float64(shift),
        scale=Float64(scale),
        construction,
    ))

filtration_kind(::Type{<:LectureTrigradeFiltration}) = :lecture_trigrade
filtration_arity(::LectureTrigradeFiltration, _data=nothing) = 3

function build_graded_complex(data::PointCloud,
                              filtration::LectureTrigradeFiltration;
                              cache::Union{Nothing,EncodingCache}=nothing)
    pts = data.points
    n = length(pts)
    c = filtration.params.center
    sh = filtration.params.shift
    sc = filtration.params.scale

    # One 0-cell per point. This keeps the example minimal and algebraically transparent.
    cells = [collect(1:n)]
    boundaries = SparseMatrixCSC{Int,Int}[]

    grades = Vector{NTuple{3,Float64}}(undef, n)
    @inbounds for i in 1:n
        x = Float64(pts[i][1])
        grades[i] = (x + sh, sc * x^2, abs(x - c))
    end

    G = GradedComplex(cells, boundaries, grades)

    # Axes are explicit and sorted by coordinate; orientation is standard (+,+,+).
    ax1 = unique(Float64[g[1] for g in grades]); sort!(ax1)
    ax2 = unique(Float64[g[2] for g in grades]); sort!(ax2)
    ax3 = unique(Float64[g[3] for g in grades]); sort!(ax3)
    axes = (ax1, ax2, ax3)
    orientation = (1, 1, 1)

    return G, axes, orientation
end


## Step 2 — Register the family and schema

The schema is the user-facing contract for `FiltrationSpec(kind=:lecture_trigrade, ...)`.

- `required`: keys that must be present in the spec,
- `defaults`: auto-filled when omitted,
- `types`: type constraints,
- `checks`: domain constraints with mathematical error messages.


In [ ]:
register_filtration_family!(;
    kind=:lecture_trigrade,
    ctor = spec -> LectureTrigradeFiltration(;
        center=Float64(spec.params[:center]),
        shift=Float64(get(spec.params, :shift, 0.0)),
        scale=Float64(get(spec.params, :scale, 1.0)),
        construction=get(spec.params, :construction, ConstructionOptions()),
    ),
    builder = build_graded_complex,
    arity = 3,
    schema = (
        required = (:center,),
        defaults = (shift=0.0, scale=1.0),
        types = (center=Real, shift=Real, scale=Real),
        checks = (scale=((x)->x > 0.0, "expected `scale > 0` for trigrade construction."),),
    ),
)

@show :lecture_trigrade in available_filtrations()
@show filtration_signature(:lecture_trigrade)
@show filtration_parameters(:lecture_trigrade)


## Step 3 — Run the typed-filtration path

Users can stay on the standard workflow entrypoint:

`encode(data, filtration; stage=:encoding_result, degree=0)`.

Other stage choices that remain available are `:simplex_tree`, `:graded_complex`, `:cochain`, `:module`, `:fringe`, `:flange`, `:cohomology_dims`.


In [ ]:
xs = range(-2.0, 2.0; length=21)
data = PointCloud([[x] for x in xs])

filt = LectureTrigradeFiltration(; center=0.0, shift=0.25, scale=1.5,
                                 construction=ConstructionOptions(output_stage=:encoding_result))

enc_typed = encode(data, filt; stage=:encoding_result, degree=0)

@show typeof(enc_typed)
@show typeof(enc_typed.M)
@show module_dims(enc_typed.M)


## Step 4 — Run the `FiltrationSpec` path (automatic roundtrip)

Because the family is registered, this works without editing internal dispatch chains.


In [ ]:
spec = FiltrationSpec(kind=:lecture_trigrade, center=0.0, shift=0.25, scale=1.5)
enc_spec = encode(data, spec; stage=:encoding_result, degree=0)

@assert module_dims(enc_typed.M) == module_dims(enc_spec.M)
println("typed/spec module dims parity: OK")


## Step 5 — Stage parity and arity checks

Use `:graded_complex` when you want to inspect the raw multigraded object directly.


In [ ]:
G_typed = encode(data, filt; stage=:graded_complex)
G_spec = encode(data, spec; stage=:graded_complex)

@assert G_typed.grades == G_spec.grades
@assert filtration_arity(filt, data) == 3
println("graded_complex parity and arity checks: OK")


## Step 6 — Invariant -> feature map

A featurizer spec defines a deterministic map from an encoding/module summary to a fixed-length vector.

For `RankGridSpec`, each feature entry corresponds to one grid location in rank-invariant sampling.
That fixed coordinate system is what makes the output ML-ready.


In [ ]:
rg = RankGridSpec((12, 12))
feat = transform(rg, enc_typed)

ri = rank_invariant(materialize_module(enc_typed.M), InvariantOptions(); store_zeros=false)

@show nfeatures(rg)
@show length(feat)
println("nonzero rank-invariant entries: ", length(ri))
println("first 8 feature values: ", feat[1:8])


## Step 7 — Contract errors are explicit

Schema/domain failures should explain the mathematical contract that was violated.


In [ ]:
try
    bad = FiltrationSpec(kind=:lecture_trigrade, center=0.0, scale=0.0)
    encode(data, bad; stage=:encoding_result)
catch err
    println(err)
end


## Notes for adding your own family

- Keep `build_graded_complex` mathematically explicit: document the grading map.
- Keep schema strict: required parameters, defaults, and domain checks.
- Add parity checks between typed and `FiltrationSpec` routes.
- If your family is expensive, benchmark both `:graded_complex` and `:encoding_result` stages.
